# Plotting a cortical mesh coloured by parcellation

Example/utility notebook showing how to load a cortical surface mesh (produced by `makeBrainMesh.ipynb`) together with a volumetric parcellation atlas, map parcel labels onto mesh vertices, and colour the mesh using proper graph colouring (adjacent parcels get maximally distinct colours, via `networkx`) rather than an arbitrary colour-per-index scheme.

In [ ]:
import nibabel as nib
import numpy as np
import trimesh
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.io import loadmat
import scipy.sparse
from scipy.spatial import cKDTree
import networkx as nx

from pathlib import Path

# ---- USER SETTINGS ----
# Root of the drive/mount holding the MRI/mesh data (the same `driveName` used
# throughout the imageReconstruction MATLAB repo). Edit for your machine -
# pathlib.Path handles the rest cross-platform, no OS-specific branching needed.
dataDriveRoot = Path('/Volumes/YourDriveName')  # <-- edit: external drive mount point, or a local/mapped folder


## Shared function

Loads an atlas + mesh pair for one age, maps parcel labels onto the mesh's vertices (filling any unlabelled vertices via nearest-neighbour), builds a parcel adjacency graph from the mesh topology, greedy-colours it, and displays the result.

In [ ]:
def plot_mesh_parcellation(atlas_path, gmvol_path, mesh_mat_path,
                            colormap_name='hsv', background=None):
    """
    Load an atlas + cortical mesh pair, colour the mesh by parcel using graph
    colouring, and display it.

    Parameters
    ----------
    atlas_path : Path or str
        Registered parcellation atlas NIfTI (e.g. the Wang432 WangFunctional_Reg_Head file).
    gmvol_path : Path or str
        Registered GM volume NIfTI, used only to sanity-check the atlas's affine
        matches (not otherwise used).
    mesh_mat_path : Path or str
        Cortical surface mesh .mat file (vertices/faces), as produced by
        makeBrainMesh.ipynb.
    colormap_name : str
        Any matplotlib colormap name.
    background : list[int] or None
        Optional RGBA background colour for the trimesh viewer (0-255 per channel).

    Returns
    -------
    mesh : trimesh.Trimesh
        The coloured mesh (already shown, but also returned for further use).
    """
    # --- Load volumetric atlas ---
    atlas_img = nib.load(str(atlas_path))
    atlas_data = atlas_img.get_fdata()
    atlas_affine = atlas_img.affine
    atlas_shape = atlas_data.shape

    unique_parcels = np.unique(atlas_data)
    print(f"Unique parcel labels: {unique_parcels}")
    print(f"Number of unique parcels: {len(unique_parcels)}")

    # --- Load GM reference volume to check affine match ---
    gmvol_img = nib.load(str(gmvol_path))
    assert np.allclose(gmvol_img.affine, atlas_affine), "WARNING: Atlas and GM volume affine matrices don't match!"

    # --- Load surface mesh from .mat ---
    mat = loadmat(str(mesh_mat_path))
    vertices = mat['vertices']  # (n, 3)
    faces = mat['faces'] - 1    # (m, 3), MATLAB is 1-based indexing; Python uses 0-based

    # --- Convert vertices to voxel space using inverse affine ---
    inv_affine = np.linalg.inv(atlas_affine)
    voxel_coords = nib.affines.apply_affine(inv_affine, vertices)
    voxel_coords = np.round(voxel_coords).astype(int)
    voxel_coords = np.clip(voxel_coords, [0, 0, 0], np.array(atlas_shape) - 1)

    # --- Sample labels at each vertex ---
    labels = np.array([atlas_data[x, y, z] for x, y, z in voxel_coords])

    # Identify which vertices are unlabelled
    unlabelled_idx = np.where(labels == 0)[0]
    labelled_idx = np.where(labels != 0)[0]

    if len(unlabelled_idx) > 0:
        print(f"Filling {len(unlabelled_idx)} unlabelled vertices...")
        tree = cKDTree(vertices[labelled_idx])
        dists, nearest = tree.query(vertices[unlabelled_idx])
        labels[unlabelled_idx] = labels[labelled_idx][nearest]
        print("Label filling complete.")
    else:
        print("No unlabelled vertices found.")

    unique_parcels = np.unique(labels)
    parcel_labels = unique_parcels[unique_parcels != 0]  # drop background (0)
    print(f"Found {len(parcel_labels)} parcels excluding background")

    # --- Build parcel adjacency graph from mesh topology ---
    n_vertices = len(vertices)
    I = faces.flatten()
    J = np.roll(faces, shift=-1, axis=1).flatten()
    adjacency = scipy.sparse.coo_matrix((np.ones(len(I)), (I, J)), shape=(n_vertices, n_vertices))
    adjacency = adjacency + adjacency.T  # symmetric
    adjacency = adjacency.tocsr()

    vertex_parcels = labels.astype(int)
    parcel_graph = nx.Graph()

    for i in range(n_vertices):
        parcel_i = vertex_parcels[i]
        if parcel_i == 0:
            continue
        for j in adjacency.indices[adjacency.indptr[i]:adjacency.indptr[i+1]]:
            parcel_j = vertex_parcels[j]
            if parcel_j == 0 or parcel_i == parcel_j:
                continue
            parcel_graph.add_edge(parcel_i, parcel_j)

    # --- Graph colouring: adjacent parcels get maximally distinct colours ---
    colouring = nx.coloring.greedy_color(parcel_graph, strategy="largest_first")

    # sort colour indices so they're compact (0...N)
    sorted_colours = {p: i for i, (p, _) in enumerate(sorted(colouring.items(), key=lambda x: x[1]))}
    parcel_to_colourindex = sorted_colours

    max_colour_index = max(parcel_to_colourindex.values()) + 1
    cmap = cm.get_cmap(colormap_name, max_colour_index)

    rgb_colors = np.array([
        cmap(parcel_to_colourindex[label])[:3] if label in parcel_to_colourindex else [0.5, 0.5, 0.5]
        for label in labels
    ])

    mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
    mesh.visual.vertex_colors = (rgb_colors * 255).astype(np.uint8)

    if background is not None:
        mesh.show(background=background)
    else:
        mesh.show()

    return mesh


## Example: 6-month atlas

In [ ]:
mesh_6mo = plot_mesh_parcellation(
    atlas_path=dataDriveRoot / 'mri' / 'registered' / 'UNC_Surf_to_NeuroDev' / 'No Mask' / '06mo' / '06mo_WangFunctional_Reg_Head.nii.gz',
    gmvol_path=dataDriveRoot / 'mri' / 'registered' / 'UNC_to_NeuroDev' / 'No Mask' / '06mo' / '06mo_GM_Reg_Head.nii.gz',
    mesh_mat_path=dataDriveRoot / 'mri' / 'meshes' / 'corticalSurface06mo.mat',
    colormap_name='hsv',
)


## Example: 12-month atlas

NOTE: this now uses the same Wang432 atlas as the 6mo example above (previously used a different, coarser atlas file - `Parc_SurfaceOnly` - which looked like an unintentional leftover, given the 6mo cell had already been updated to Wang432 and every other part of this project uses Wang432 throughout).

In [ ]:
mesh_12mo = plot_mesh_parcellation(
    atlas_path=dataDriveRoot / 'mri' / 'registered' / 'UNC_Surf_to_NeuroDev' / 'No Mask' / '12mo' / '12mo_WangFunctional_Reg_Head.nii.gz',
    gmvol_path=dataDriveRoot / 'mri' / 'registered' / 'UNC_to_NeuroDev' / 'No Mask' / '12mo' / '12mo_GM_Reg_Head.nii.gz',
    mesh_mat_path=dataDriveRoot / 'mri' / 'meshes' / 'corticalSurface12mo.mat',
    colormap_name='nipy_spectral',
    background=[0, 255, 255, 255],
)
